# Lakebase Search customer-support demo

This notebook walks through the same hybrid search used by the chat app. It validates exact order-ID lookup, natural-language issue search, customer/status filters, and the Lakebase SQL path.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'demo').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Working directory:', ROOT)
print('Lakebase DSN configured:', bool(os.getenv('LAKEBASE_DSN')))

## 1. Run the local fallback first

The local fallback uses the same demo schema and search behavior without requiring network access or a Lakebase project. Its deterministic embedding is only a stand-in for a production embedding model.

In [ ]:
from demo.search import LocalDemoStore

local = LocalDemoStore()
local.seed()

queries = [
    ('ORDER-BRAVO', 'customer-a'),
    ('shoes arrived damaged', 'customer-a'),
    ('tracking says delivered but nothing came', 'customer-a'),
]
for query, customer_id in queries:
    print('\nQUERY:', query)
    for result in local.search(query, customer_id=customer_id):
        print(result.order_id, result.status, result.match_type, result.score)

## 2. Lakebase setup

Enable Lakebase Search in the project settings before running the SQL. Then run `lakebase_setup.sql` against the Lakebase database. The setup installs `lakebase_vector` and `lakebase_text` and creates the operational table.

In [ ]:
# In a Databricks notebook, install the client if it is not already available.
# %pip install 'psycopg[binary]>=3.2,<4'

from demo.data import DEMO_CASES
from demo.search import LakebaseSearchStore

if not os.getenv('LAKEBASE_DSN'):
    print('Set LAKEBASE_DSN to run the Lakebase cells.')
else:
    lakebase = LakebaseSearchStore()
    lakebase.seed(DEMO_CASES)
    print('Seeded Lakebase and created the ANN and BM25 indexes.')

## 3. Run hybrid search in Lakebase

The Lakebase implementation runs vector and BM25 searches separately, applies the customer/status filter in the database, and combines the ranked candidates with Reciprocal Rank Fusion.

In [ ]:
if os.getenv('LAKEBASE_DSN'):
    for query in ['ORDER-BRAVO', 'shoes arrived damaged', 'tracking says delivered but nothing came']:
        print('\nQUERY:', query)
        for result in lakebase.search(query, customer_id='customer-a'):
            print(result.order_id, result.status, result.match_type, result.score)
else:
    print('Lakebase cells skipped because LAKEBASE_DSN is not configured.')

## 4. Start the chat app

From a terminal in the package root, run `python -m demo.app`. Use `SEARCH_BACKEND=lakebase` to point the UI at Lakebase; omit it for local mode.

In [ ]:
print('Run in a terminal:')
print('  python -m demo.app')
print('or:')
print('  SEARCH_BACKEND=lakebase python -m demo.app')